### Exporting configuration and repo root

In [0]:
import sys
from pathlib import Path

current = Path.cwd()

repo_root = next(
    (
        path
        for path in [current, *current.parents]
        if (
            path / "src" / "house_price_simple_analysis"
        ).is_dir()
    ),
    None,
)

if repo_root is None:
    raise FileNotFoundError(
        f"Project root not found from {current}"
    )

src_path = str(repo_root / "src")

if src_path not in sys.path:
    sys.path.insert(0, src_path)

from house_price_simple_analysis.config import ProjectConfig

config = ProjectConfig.from_yaml(
    str(repo_root / "config" / "project_config.yml")
)

### Load the champion model (the one in production) and the model logged in the last running test.

In [0]:
import mlflow
import mlflow.sklearn

mlflow.set_registry_uri("databricks-uc")

model_name = (
    f"{config.databricks.catalog_name}."
    f"{config.databricks.schema_name}."
    f"{config.model.registered_model_name}"
)

# Existing approved model
champion_uri = f"models:/{model_name}@Champion"

# Replace with the model URI from your latest training run
candidate_uri = "models:/m-612a9b324ad44b5895736c4718c70d10"

champion = mlflow.sklearn.load_model(champion_uri)
candidate = mlflow.sklearn.load_model(candidate_uri)

### Load the evaluation dataset

In [0]:
from sklearn.metrics import mean_squared_error
import numpy as np

test_table = (
    f"{config.databricks.catalog_name}."
    f"{config.databricks.schema_name}."
    f"{config.databricks.test_table}"
)

test_df = spark.table(test_table).toPandas()

features = config.data.num_features
target = config.data.target

X_test = test_df[features].astype("float64")
y_test = test_df[target]

### Compare model performance

In [0]:
champion_predictions = champion.predict(X_test)
candidate_predictions = candidate.predict(X_test)

champion_rmse = np.sqrt(
    mean_squared_error(y_test, champion_predictions)
)

candidate_rmse = np.sqrt(
    mean_squared_error(y_test, candidate_predictions)
)

improvement = (
    (champion_rmse - candidate_rmse) / champion_rmse
)

print(f"Champion RMSE: {champion_rmse:.2f}")
print(f"Candidate RMSE: {candidate_rmse:.2f}")
print(f"Improvement: {improvement:.2%}")

### Introduce a promotion rule

In [0]:
minimum_improvement = 0.01

if champion_rmse <= 0:
    raise ValueError(
        "Champion RMSE must be positive for relative comparison."
    )

approved = improvement >= minimum_improvement

if approved:
    print("Candidate passed validation.")
    print("Eligible for registration and promotion.")
else:
    print("Candidate did not pass validation.")
    print("Keep the current Champion.")